In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.iloc[1][0]

/tmp/ipykernel_5569/1019881219.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.iloc[1][0]


'“It is our choices, Harry, that show what we truly are, far more than our abilities.”'

In [5]:
df.shape

(3038, 2)

In [6]:
df.isnull().sum()

,0
quote,0
Author,0


In [7]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [8]:
quotes = quotes.str.lower()

In [9]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [10]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 10000
tok = Tokenizer(num_words=vocab_size)
tok.fit_on_texts(quotes)

In [13]:
word_index = tok.word_index
print(len(word_index))
print(list(word_index.items())[:10])

8979
[('<OOV>', 1), ('the', 2), ('you', 3), ('to', 4), ('and', 5), ('a', 6), ('i', 7), ('is', 8), ('of', 9), ('that', 10)]


In [14]:
sequences = tok.texts_to_sequences(quotes)

print(quotes[0])
print(sequences[0])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
[714, 63, 30, 20, 17, 947, 11, 8, 6, 1157, 9, 71, 294, 11, 146, 13, 810, 105, 753, 71, 2462]


In [15]:
X = []
y = []

for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

In [16]:
X[17]

[714, 63, 30, 20, 17, 947, 11, 8, 6, 1157, 9, 71, 294, 11, 146, 13, 810, 105]

In [17]:
X[0]

[714]

In [18]:
y[0]

63

In [19]:
len(X), len(y)

(85271, 85271)

In [20]:
max_len = max(len(x) for x in X)
max_len

745

In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [22]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [23]:
X_padded.shape

(85271, 745)

In [24]:
y = np.array(y)
y.shape

(85271,)

In [25]:
from tensorflow.keras.utils import to_categorical
y_encoded = to_categorical(y, num_classes=vocab_size)

In [26]:
y.shape, y_encoded.shape, y_encoded[0]

((85271,), (85271, 10000), array([0., 0., 0., ..., 0., 0., 0.]))

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [28]:
Embedding_dim = 64
rnn_units = 128

In [29]:
rnn_model = Sequential()

rnn_model.add(Embedding(input_dim=vocab_size, output_dim=Embedding_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [30]:
rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [31]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=Embedding_dim, input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [33]:
lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [34]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
epochs = 100
batch_size = 128

In [ ]:
history_lstm = lstm_model.fit(X_padded, y_encoded, epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 79ms/step - accuracy: 0.1233 - loss: 5.4098 - val_accuracy: 0.1072 - val_loss: 6.3395
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 45s 75ms/step - accuracy: 0.1361 - loss: 5.1676 - val_accuracy: 0.1091 - val_loss: 6.3999
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 78ms/step - accuracy: 0.1492 - loss: 4.9367 - val_accuracy: 0.1093 - val_loss: 6.4710
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 45s 75ms/step - accuracy: 0.1642 - loss: 4.7159 - val_accuracy: 0.1099 - val_loss: 6.5425
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 45s 75ms/step - accuracy: 0.1812 - loss: 4.5038 - val_accuracy: 0.1128 - val_loss: 6.6120
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.2017 - loss: 4.3043 - val_accuracy: 0.1108 - val_loss: 6.6863
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.2239 - loss: 4.1139 - val_accuracy: 0.1079 - val_loss: 6.7651
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.2473 - loss: 3

In [39]:
lstm_model.save("lstm_model.h5")

In [40]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:
def predictor(model, tokenizer, text, max_len):
  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')
  pred = model.predict(seq, verbose=0)
  valid_predictions = pred[0][1:max(index_to_word.keys()) + 1]
  pred_index = np.argmax(valid_predictions)
  return index_to_word[pred_index]

In [74]:
seed = "where are you"
next_word = predictor(lstm_model, tok, seed, max_len)
print(seed + " " + next_word)

where are you ever


In [ ]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tok, f)

In [ ]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)